# Synthetic Query Development

### Maxime Bouthillier

### Univeristy of Waterloo

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM
logging.set_verbosity_error()
import numpy as np
import os
from torch import Tensor
from torch.utils.data import DataLoader
import faiss
import json
from beir.datasets.data_loader import GenericDataLoader
from tqdm import tqdm
from dotenv import load_dotenv
import os
import re

/work/mbouthil/.conda/envs/myuwenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Loading Data
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [ ]:
print('Total unique queries:', len(queries))
print('Total passages:', len(corpus))

Total unique queries: 502939
Total passages: 8841823


In [ ]:
x = len(queries)

In [ ]:
queries[str(x)]

'state constitutions tended to __________'

In [ ]:
qrels[str(x)]

{'769709': 1}

In [ ]:
len(corpus)

8841823

# Additional Synthetic Query Generation

In [ ]:
N = 10,000

In [ ]:
q_id, test_query = [(keys, values) for keys, values in queries.items()][0]
print(test_query)
print(q_id)

)what was the immediate impact of the success of the manhattan project?
1185869


In [ ]:
qrels[q_id]

{'0': 1}

In [ ]:
for keys, values in qrels.items():
    print(keys)
    print(values)
    break

1185869
{'0': 1}


### LLM

In [2]:
# Authenticating Token
load_dotenv('/work/mbouthil/MMATH-CM-Research-Project/token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

In [3]:
from huggingface_hub import login
login(token=token)

In [4]:
# Loading model and Tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token
)

Loading weights: 100%|██████████| 291/291 [00:01<00:00, 150.69it/s, Materializing param=model.norm.weight]                              


In [ ]:
system_prompt = '''
You are a helpful AI Assistant. You are to follow the following instructions:

You will be given a query. Your task is to create a new query that asks the same questions as
the provided query, however, posed differently. Provide only the new query and format is as follows:

**new query** 
'''

In [ ]:
def llm_call(notes: list[str], system_prompt:str) -> list[str]:
    messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note}
        ]
        for note in notes
    ]

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_p=0.9,
            do_sample=True
        )

    responses = []
    for i in range(len(notes)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

In [ ]:
response = llm_call([test_query], system_prompt)

In [ ]:
print(response[0])

assistant

**What were the direct consequences of the Manhattan Project's achievement?**


In [ ]:
new_query = re.findall(r'\*\*([^*]+)\*\*', response[0])[0]
print(new_query)

What were the direct consequences of the Manhattan Project's achievement?


In [ ]:
print(test_query)

)what was the immediate impact of the success of the manhattan project?


In [ ]:
qrels[q_id]

{'0': 1}

## Updating qrels

In [ ]:
t_query[str(len(t_query)+1)] = new_query
qrels[str(len(t_query))] = qrels[q_id]

In [ ]:
qrels[str(len(t_query))]

{'0': 1}

In [ ]:
t_query[str(len(t_query))]

"What were the direct consequences of the Manhattan Project's achievement?"

# All Together

### Loading Preambles

In [2]:
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [3]:
t_corpus = corpus.copy()
t_queries = queries.copy()
t_qrels = qrels.copy()
max_id = int(max([int(key) for key in t_queries.keys()]))

In [4]:
N = 10_000
q_subset = [(keys, values) for keys, values in queries.items()][:N]

In [5]:
# Authenticating Token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model and Tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
system_prompt = '''
You are a helpful AI Assistant. You are to follow the following instructions:

You will be given a query. Your task is to create a new query that asks the same questions as
the provided query, however, posed differently. Provide only the new query and format is as follows:

**new query** 
'''

In [7]:
def llm_call(notes: list[str], system_prompt:str) -> list[str]:
    messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note}
        ]
        for note in notes
    ]

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_p=0.9,
            do_sample=True
        )

    responses = []
    for i in range(len(notes)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

### Genearting New Queries

In [9]:
# Creating Batches
def batch_splits(queries:list, batch_size:int=64):
    for i in range(0, len(queries), batch_size):
        yield queries[i:i + batch_size]

batches = batch_splits(q_subset)

In [ ]:
new_queries = []
old_ids = []
i = 1

for batch in batches:

    ids = [id[0] for id in batch]
    queries = [id[1] for id in batch]

    queries = llm_call(queries, system_prompt)
    queries = [
        re.findall(r'\*\*([^*]+)\*\*', query)[0] 
        if len(re.findall(r'\*\*([^*]+)\*\*', query)) != 0 
        else query 
        for query in queries
        ]

    
    new_queries.extend(queries)
    old_ids.extend(ids)
    if i == 1:
        break
    i += 1

new_q_id = max_id + 1 


# Expanding Dataset
for i, id in enumerate(old_ids):

    t_queries[new_q_id] = new_queries[i]            # Creating new query
    t_qrels[new_q_id] = t_qrels[id]                 # Mapping new query to positive passages
    new_q_id += 1

In [11]:
print(len(new_queries))

64


In [17]:
new_q_id = max_id + 1 

for i, id in enumerate(old_ids):

    t_queries[new_q_id] = new_queries[i]              # Creating new query
    t_qrels[new_q_id] = t_qrels[id]                 # Mapping new query to positive passages
    new_q_id += 1

In [24]:
for key in t_qrels.keys():
    if key == 1185870:
        print(key)

1185870


In [25]:
for key in t_queries.keys():
    if key == 1185870:
        print(key)

1185870


In [19]:
len(t_queries)

503003

In [18]:
print(len(qrels))
print(len(t_qrels))

502939
503003


### Saving new dataset

In [ ]:
import json
import os
import shutil

# Setup paths
original_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
modified_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco_modified"

# Create directories
os.makedirs(modified_dir, exist_ok=True)
os.makedirs(os.path.join(modified_dir, "qrels"), exist_ok=True)

# 1. Copy old corpus (since it's unchanged)
print("Copying corpus...")
shutil.copy(
    f"{original_dir}/corpus.jsonl", 
    f"{modified_dir}/corpus.jsonl"
)
print(f"✓ Copied corpus")

print("Saving modified queries...")
queries_path = os.path.join(modified_dir, "queries.jsonl")
with open(queries_path, 'w') as f:
    for query_id, query_text in t_queries.items():
        entry = {"_id": str(query_id), "text": query_text}  # Force string
        f.write(json.dumps(entry) + '\n')
print(f"✓ Saved {len(t_queries)} queries")

# 3. Save new qrels
print("Saving modified qrels...")
qrels_path = os.path.join(modified_dir, "qrels", "train.tsv")
with open(qrels_path, 'w') as f:
    f.write("query-id\tcorpus-id\tscore\n")
    for query_id, doc_scores in t_qrels.items():
        for doc_id, score in doc_scores.items():
            f.write(f"{str(query_id)}\t{str(doc_id)}\t{score}\n")  # Force string
print(f"✓ Saved {len(t_qrels)} qrels")

print(f"\n✓ Complete dataset saved to {modified_dir}")

Copying corpus...
✓ Copied corpus
Saving modified queries...
✓ Saved 503003 queries
Saving modified qrels...

✓ Complete dataset saved to /work/mbouthil/projects/research_project/RAG/datasets/msmarco_modified


In [27]:
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco_modified"
new_corpus, new_queries, new_qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [30]:
print(len(new_queries))

503003


In [32]:
print('It worked!')

It worked!
